Comparação de métodos de classificação de ML

* KNN (variar K, ponderação)
* AD
* MLP (nº de camadas, função de atv e tax de aprendizado)
* SVM
* Naive bayes
* Randon Forest
* Begging (Default)
* Boosting (Default)

In [ ]:
import pandas as pd
import sklearn as sk
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import warnings

from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import AdaBoostClassifier
from scipy.stats import rankdata

import warnings



ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
df = pd.read_csv('heart.csv')
df = shuffle(df)

In [ ]:
x_treino,x_temp,y_treino,y_temp=train_test_split(df.drop('target',axis=1),df['target'],test_size=0.5, stratify = df['target'])
x_validacao,x_teste,y_validacao,y_teste=train_test_split(x_temp,y_temp,test_size=0.5, stratify = y_temp)

print("Treino")
x_treino.info()
y_treino.info()

print("\nValidação")
x_validacao.info()
y_validacao.info()

print("\nTeste")
x_teste.info()
y_teste.info()

Treino
<class 'pandas.DataFrame'>
Index: 512 entries, 811 to 500
Data columns (total 13 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   age       512 non-null    int64  
 1   sex       512 non-null    int64  
 2   cp        512 non-null    int64  
 3   trestbps  512 non-null    int64  
 4   chol      512 non-null    int64  
 5   fbs       512 non-null    int64  
 6   restecg   512 non-null    int64  
 7   thalach   512 non-null    int64  
 8   exang     512 non-null    int64  
 9   oldpeak   512 non-null    float64
 10  slope     512 non-null    int64  
 11  ca        512 non-null    int64  
 12  thal      512 non-null    int64  
dtypes: float64(1), int64(12)
memory usage: 56.0 KB
<class 'pandas.Series'>
Index: 512 entries, 811 to 500
Series name: target
Non-Null Count  Dtype
--------------  -----
512 non-null    int64
dtypes: int64(1)
memory usage: 8.0 KB

Validação
<class 'pandas.DataFrame'>
Index: 256 entries, 962 to 404
Data columns (tot

# **KNN**

In [ ]:
def ponder_1(distances):
    max_dist = np.max(distances, axis=1, keepdims=True)

    with np.errstate(divide='ignore', invalid='ignore'):
        weights = 1.0 - (distances / max_dist)

    weights = np.where(max_dist == 0, 1.0, weights)

    soma_linha = np.sum(weights, axis=1, keepdims=True)
    weights = np.where(soma_linha == 0, 1.0, weights)

    return weights

In [ ]:
maiorAcc = -1
for j in ("distance","uniform", ponder_1): #ponderação
  for i in range (1,50): #nº de vizinhos
    KNN = KNeighborsClassifier(n_neighbors=i,weights=j)
    KNN.fit(x_treino,y_treino)
    opiniao = KNN.predict(x_validacao)
    Acc = accuracy_score(y_validacao, opiniao)
    if (Acc > maiorAcc):
      maiorAcc = Acc
      melhor_modelo_KNN = KNN

In [ ]:
opiniao = melhor_modelo_KNN.predict(x_teste)
print("\nK: ",melhor_modelo_KNN.n_neighbors, "; Ponderação:", melhor_modelo_KNN.weights, "\n Acurácia sobre o teste: ",accuracy_score(y_teste, opiniao))


K:  32 ; Ponderação: distance 
 Acurácia sobre o teste:  0.9455252918287937


# **Árvore de decisão**

In [ ]:
arvore = DecisionTreeClassifier()

maior = -1
for j in ("entropy","gini"):  #criterion
  for i in range (1,11):      #max_depth
    for k in range (1,11):    #min_samples_leaf
      for l in range (2,16):  #min_samples_split
        for m in ('best','random'): #splitter
          AD = DecisionTreeClassifier(criterion=j,max_depth=i,min_samples_leaf=k,min_samples_split=l,splitter=m)
          AD.fit(x_treino,y_treino)
          opiniao = AD.predict(x_validacao)
          Acc = accuracy_score(y_validacao, opiniao)
          if (Acc > maior):
            maior = Acc
            melhor_modelo_AD = AD

In [ ]:
opiniao = melhor_modelo_AD.predict(x_teste)
print("melhor configuração: Criterion: ",melhor_modelo_AD.criterion," max_depth: ",melhor_modelo_AD.max_depth," min_samples_leaf: ",melhor_modelo_AD.min_samples_leaf," min_samples_split: ",melhor_modelo_AD.min_samples_split," splitter: ",melhor_modelo_AD.splitter," Acc: ",maior)
print("\nAcurácia sobre o teste: ",accuracy_score(y_teste, opiniao))

melhor configuração: Criterion:  gini  max_depth:  7  min_samples_leaf:  1  min_samples_split:  5  splitter:  best  Acc:  0.97265625

Acurácia sobre o teste:  0.953307392996109


# **Naive bayes**

In [ ]:
maior = -1

NB = GaussianNB()
NB.fit(x_treino, y_treino)

opiniao = NB.predict(x_validacao)
Acc = accuracy_score(y_validacao, opiniao)

melhor_nb = NB

print("Acc: ", Acc)

opiniao = NB.predict(x_teste)
print("Acurácia sobre o teste: ",accuracy_score(y_teste, opiniao))

Acc:  0.78515625
Acurácia sobre o teste:  0.7937743190661478


# **SVM**

In [ ]:
maiorAcc = -1
melhor_svm = None

lista_C = [0.1, 1, 10, 100]
lista_kernel = ['linear', 'rbf', 'sigmoid']

total_iteracoes = len(lista_C) * len(lista_kernel)

for c in lista_C:
  for k in lista_kernel:
    svm = SVC(C=c, kernel=k, probability=True, random_state=42)
    svm.fit(x_treino, y_treino)

    opiniao = svm.predict(x_validacao)
    acc = accuracy_score(y_validacao, opiniao)

    if acc > maiorAcc:
        maiorAcc = acc
        melhor_svm = svm

In [ ]:
opiniao_teste_svm = melhor_svm.predict(x_teste)
acc_teste_svm = accuracy_score(y_teste, opiniao_teste_svm)

print(f"Configuração: C={melhor_svm.C}, kernel={melhor_svm.kernel}, gamma={melhor_svm.gamma}")
print(f"Acurácia final no conjunto de Teste: {acc_teste_svm:.4f}")

Configuração: C=100, kernel=linear, gamma=scale
Acurácia final no conjunto de Teste: 0.8482


# **MLP**

In [ ]:
hidden_layers = []
for n_layers in range(1, 3):
    for n_neurons in range(13, 27, 2):
        hidden_layers.append((n_neurons,) * n_layers)

param_grid = {
    'hidden_layer_sizes': hidden_layers,
    'activation': ['tanh', 'relu', 'logistic'],
    'learning_rate_init': [0.0001, 0.001, 0.01, 0.1],
    'max_iter': [100, 500, 1000],
    'batch_size': [16, 32, 64]
}

maiorAcc = -1
melhor_mlp = None

for params in ParameterGrid(param_grid):
    mlp = MLPClassifier(**params, random_state=42)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        mlp.fit(x_treino, y_treino)

    opiniao = mlp.predict(x_validacao)
    Acc = accuracy_score(y_validacao, opiniao)

    if Acc > maiorAcc:
        maiorAcc = Acc
        melhor_mlp = mlp

print(f"Melhor configuração: {params['max_iter']} | {params['hidden_layer_sizes']} | Act: {params['activation']} | LR: {params['learning_rate_init']} | Batch: {params['batch_size']} | Acc: {Acc:.4f}")

Melhor configuração: 1000 | (25, 25) | Act: logistic | LR: 0.1 | Batch: 64 | Acc: 0.5117


In [ ]:
opiniao_teste = melhor_mlp.predict(x_teste)
acc_teste = accuracy_score(y_teste, opiniao_teste)
print(f"\nAcurácia final no Teste: {acc_teste:.4f}")


Acurácia final no Teste: 0.9222


# **Bagging (Default)**

In [ ]:
models = [KNeighborsClassifier(), SVC(), GaussianNB(), MLPClassifier(), DecisionTreeClassifier()]
max_sample = [0.2, 0.4, 0.6, 0.8]
search_estimators = [10, 50, 100]

melhor_acuracia = -1
melhor_config = ""

for base_model in models:
    nome_base = base_model.__class__.__name__
    for n in search_estimators:
        for s in max_sample:
            bagg = BaggingClassifier(estimator=base_model, n_estimators=n, max_samples=s, random_state=42)

            # sem warnings!
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                bagg.fit(x_treino, y_treino)
                resposta = bagg.predict(x_validacao)

            acc = accuracy_score(y_validacao, resposta)


            if acc > melhor_acuracia:
                melhor_acuracia = acc
                melhor_config = f"{nome_base} com {n} estimadores"
                melhor_modelo_bagging = bagg

print(f"\nMelhor Configuração: {melhor_config} (Acurácia: {melhor_acuracia:.4f})")

y_pred = melhor_modelo_bagging.predict(x_teste)
print('Acurácia no teste: {0:0.4f}'.format(accuracy_score(y_teste, y_pred)))


Melhor Configuração: DecisionTreeClassifier com 50 estimadores (Acurácia: 0.9492)
Acurácia no teste: 0.9572


# **Randon Forest**

In [ ]:
param_grid = {
    'n_estimators': [50, 100, 150],
    'criterion': ['gini', 'entropy'],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}


maiorAcc = -1
melhor_RF = None

for params in ParameterGrid(param_grid):
    RF = RandomForestClassifier(**params, random_state=42)
    RF.fit(x_treino, y_treino)
    opiniao = RF.predict(x_validacao)
    Acc = accuracy_score(y_validacao, opiniao)

    if Acc > maiorAcc:
        maiorAcc = Acc
        melhor_RF = RF

y_pred = melhor_RF.predict(x_teste)

print("Melhor configuração:")
print(melhor_RF.get_params())
print('Acurácia no teste: {0:0.4f}'.format(accuracy_score(y_teste, y_pred)))

Melhor configuração:
{'bootstrap': True, 'ccp_alpha': 0.0, 'class_weight': None, 'criterion': 'gini', 'max_depth': 10, 'max_features': 'sqrt', 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 2, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 50, 'n_jobs': None, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
Acurácia no teste: 0.9728


# **Boosting (Default)**

In [ ]:
models = [SVC(), GaussianNB(), DecisionTreeClassifier()]

search_estimators = [50, 100, 200]
search_learning_rate = [0.0001, 0.001, 0.01, 0.1]

melhor_acuracia = -1
melhor_config = ""

for base_model in models:
    nome_base = base_model.__class__.__name__
    for n in search_estimators:
        for lr in search_learning_rate:
            boost = AdaBoostClassifier(estimator=base_model, n_estimators=n, learning_rate=lr, random_state=42)

            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    boost.fit(x_treino, y_treino)
                    resposta = boost.predict(x_validacao)

                acc = accuracy_score(y_validacao, resposta)


                if acc > melhor_acuracia:
                    melhor_acuracia = acc
                    melhor_config = f"{nome_base} com {n} estimadores e learning_rate={lr}"
                    melhor_modelo_boosting = boost

            except Exception as e:
                print(f"Base: {nome_base} | Estimadores: {n} | Learning Rate: {lr} | ERRO: {e}")

print(f"\nMelhor Configuração: {melhor_config} (Acurácia: {melhor_acuracia:.4f})")

y_pred = melhor_modelo_boosting.predict(x_teste)
print('Acurácia no teste: {0:0.4f}'.format(accuracy_score(y_teste, y_pred)))


Melhor Configuração: DecisionTreeClassifier com 50 estimadores e learning_rate=0.0001 (Acurácia: 0.9570)
Acurácia no teste: 0.9494


# **Combinação**

In [ ]:
modelos = [melhor_modelo_KNN, melhor_modelo_AD, melhor_mlp, melhor_svm, melhor_nb]
nomes_modelos = ["KNN", "AD", "MLP", "SVM", "NB"]

classes = modelos[0].classes_
n_classes = len(classes)
n_amostras = x_teste.shape[0]

probs_modelos = [modelo.predict_proba(x_teste) for modelo in modelos]
preds_modelos = [modelo.predict(x_teste) for modelo in modelos]

soma_probs = np.zeros((n_amostras, n_classes))

for probs in probs_modelos:
    soma_probs += probs

pred_soma = classes[np.argmax(soma_probs, axis=1)]
acc_soma = accuracy_score(y_teste, pred_soma)

print("Acurácia - Regra da Soma:", acc_soma)

predicoes = np.array(preds_modelos).T
pred_majoritario = []

for linha in predicoes:
    valores, contagens = np.unique(linha, return_counts=True)
    pred_majoritario.append(valores[np.argmax(contagens)])

pred_majoritario = np.array(pred_majoritario)
acc_majoritario = accuracy_score(y_teste, pred_majoritario)

print("Acurácia - Voto Majoritário:", acc_majoritario)

borda_scores = np.zeros((n_amostras, n_classes))

for probs in probs_modelos:
    for i in range(n_amostras):
        ranking = np.argsort(probs[i])  
        pontos = np.zeros(n_classes)

        for pos, classe_idx in enumerate(ranking):
            pontos[classe_idx] = pos

        borda_scores[i] += pontos

pred_borda = classes[np.argmax(borda_scores, axis=1)]
acc_borda = accuracy_score(y_teste, pred_borda)

print("Acurácia - Borda Count:", acc_borda)

Acurácia - Regra da Soma: 0.9494163424124513
Acurácia - Voto Majoritário: 0.9416342412451362
Acurácia - Borda Count: 0.9416342412451362
